# Scorer training
In this notebook we train the scorer model.  Much like for the filter training, we first embed using `FinBERT` and then cross-validate the following models:
- a simple logistic regression (baseline),
- a cosine-kNN model,
- a shallow neural network with 1 hidden layers with 8/16/32 nodes.

### Data loading and train-test split
We load the relevant scored headlines

In [1]:
import pandas as pd

df = pd.read_csv("data/scored_headlines.csv", index_col=False)
df = df[df.relevant == 1]

We embed them

In [2]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("ProsusAI/finbert")
df["embeddings"] = embedder.encode(df.concat.to_list(), show_progress_bar=True).tolist()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ProsusAI/finbert
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/31 [00:00<?, ?it/s]

We then create a new column (`jittery`) taking values:
- 1 if `score > 0.8`,
- 0 if `score < 0.5`,
- -1 otherwise.

Then, we discard all rows with `jittery == -1`.

In [3]:
df["jittery"] = df["score"].apply(lambda s: 1 if s > 0.8 else (0 if s < 0.5 else -1))
df = df[df["jittery"] != -1].reset_index(drop=True)

We create the train-test split.  Note that we will fit the score directly.

In [10]:
from sklearn.model_selection import train_test_split
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    df.embeddings, df.jittery, shuffle=True, random_state=42
)

# dataframe versions for sklearn
X_train_df = pd.DataFrame(X_train.to_list())
X_test_df = pd.DataFrame(X_test.to_list())

# numpy version for keras
X_train_np = np.stack(X_train.values)
X_test_np = np.stack(X_test.values)
y_train_np = y_train.values
y_test_np = y_test.values

### Models definitions
##### Continuous logistic regression with Ridge regularization (baseline)

In [5]:
from sklearn.linear_model import LogisticRegressionCV

logit = LogisticRegressionCV(cv=5)

##### $k$-nearest neighbors

In [6]:
from sklearn.neighbors import KNeighborsClassifier

neigh = KNeighborsClassifier(n_neighbors=5, metric="cosine", weights="distance")

##### Shallow neural networks

In [7]:
import os

os.environ["KERAS_BACKEND"] = "torch"

import keras


def generate_shallow(nodes: int, summarize: bool = True) -> keras.Model:
    inputs = keras.Input(shape=(embedder.config.hidden_size,))
    x = keras.layers.Dense(nodes, activation="relu")(inputs)
    x = keras.layers.Dropout(0.5)(x)
    outputs = keras.layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name=f"scorer ({nodes} nodes)")
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

    if summarize:
        model.summary()

    return model

In [8]:
nn8 = generate_shallow(8)
nn16 = generate_shallow(16)
nn32 = generate_shallow(32)
nn64 = generate_shallow(64)

Model: "scorer (8 nodes)"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │         6,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,161 (24.07 KB)

 Trainable params: 6,161 (24.07 KB)

 Non-trainable params: 0 (0.00 B)

...

Model: "scorer (64 nodes)"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │        49,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 49,281 (192.50 KB)

 Trainable params: 49,281 (192.50 KB)

 Non-trainable params: 0 (0.00 B)

### Model selection

In [28]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

classification_threshold = 0.65

callback = keras.callbacks.EarlyStopping(patience=2, monitor="accuracy")

kf = KFold(n_splits=5, shuffle=True, random_state=42)

results = []

for train_index, holdout_index in kf.split(X_train):
    # pandas
    X_tt_df = X_train_df.iloc[train_index]
    y_tt_df = y_train.iloc[train_index]
    X_th_df = X_train_df.iloc[holdout_index]
    y_th_df = y_train.iloc[holdout_index]

    # numpy
    X_tt_np = X_train_np[train_index]
    X_th_np = X_train_np[holdout_index]
    y_tt_np = y_train_np[train_index]
    y_th_np = y_train_np[holdout_index]

    scores = {}

    # logistic regression
    logit.fit(X_tt_df, y_tt_df)
    scores["logit"] = accuracy_score(
        y_th_df,
        (logit.predict_proba(X_th_df)[:, 1] > classification_threshold).astype(int),
    )

    # kNN
    neigh.fit(X_tt_df, y_tt_df)
    scores["kNN"] = accuracy_score(
        y_th_df,
        (neigh.predict_proba(X_th_df)[:, 1] > classification_threshold).astype(int),
    )

    # nn8
    nn8.fit(X_tt_np, y_tt_np, batch_size=64, epochs=15, callbacks=[callback], verbose=0)
    scores["nn8"] = accuracy_score(
        y_th_np, (nn8.predict(X_th_np) > classification_threshold).astype(int)
    )

    # nn16
    nn16.fit(
        X_tt_np, y_tt_np, batch_size=64, epochs=15, callbacks=[callback], verbose=0
    )
    scores["nn16"] = accuracy_score(
        y_th_np,
        (nn16.predict(X_th_np, verbose=0) > classification_threshold).astype(int),
    )

    # nn32
    nn32.fit(
        X_tt_np, y_tt_np, batch_size=64, epochs=15, callbacks=[callback], verbose=0
    )
    scores["nn32"] = accuracy_score(
        y_th_np,
        (nn32.predict(X_th_np, verbose=0) > classification_threshold).astype(int),
    )

    # nn64
    nn64.fit(
        X_tt_np, y_tt_np, batch_size=64, epochs=15, callbacks=[callback], verbose=0
    )
    scores["nn64"] = accuracy_score(
        y_th_np,
        (nn64.predict(X_th_np, verbose=0) > classification_threshold).astype(int),
    )

    results.append(scores)

results = pd.DataFrame(results)

/home/roberto/Projects/AI/jitter/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/home/roberto/Projects/AI/jitter/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


/home/roberto/Projects/AI/jitter/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/home/roberto/Projects/AI/jitter/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


/home/roberto/Projects/AI/jitter/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/home/roberto/Projects/AI/jitter/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


/home/roberto/Projects/AI/jitter/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/home/roberto/Projects/AI/jitter/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


/home/roberto/Projects/AI/jitter/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/home/roberto/Projects/AI/jitter/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


In [29]:
results

,logit,kNN,nn8,nn16,nn32,nn64
0,0.863158,0.800000,0.884211,0.915789,0.915789,0.989474
1,0.755319,0.776596,0.904255,0.914894,0.978723,0.989362
2,0.882979,0.797872,0.946809,0.957447,0.957447,1.000000
3,0.829787,0.787234,0.957447,0.914894,0.957447,0.989362
4,0.808511,0.840426,0.925532,0.914894,0.936170,0.989362


In [30]:
summary = pd.concat(
    [results.mean(), results.std()], keys=["average accuracy", "accuracy std"], axis=1
)
summary

,average accuracy,accuracy std
logit,0.827951,0.049810
kNN,0.800426,0.024231
nn8,0.923651,0.030057
nn16,0.923583,0.018934
nn32,0.949115,0.023946
nn64,0.991512,0.004745


We compare `nn16` to the baseline `logit`.

In [34]:
# logit
logit.fit(X_train_df, y_train)

# nn16
nn16.fit(
    X_train_np, y_train_np, batch_size=64, epochs=15, callbacks=[callback], verbose=0
)

# summary
print()
print(
    "Logit accuracy score:",
    accuracy_score(
        y_test,
        (logit.predict_proba(X_test_df)[:, 1] > classification_threshold).astype(int),
    ),
)
print(
    "nn16 accuracy score: ",
    accuracy_score(
        y_test_np,
        (nn16.predict(X_test_np, verbose=0) > classification_threshold).astype(int),
    ),
)

/home/roberto/Projects/AI/jitter/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/home/roberto/Projects/AI/jitter/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(



Logit accuracy score: 0.802547770700637
nn16 accuracy score:  0.8471337579617835


For the moment, this could be good enough.  In future upgrades, it might be worth binning the scores into 3 or more bins.